In [6]:
import numpy as np
import json
from pathlib import Path

In [18]:
N = 1_000_000          # liczba rekordów
FEATURES = 12          # liczba cech
NOISE = 0.01           # +/- 0.01 jak w StreamGen
SEED = 42              # dla powtarzalności

np.random.seed(SEED)

In [38]:
# Generacja danych jak StreamGen

# losowanie parametrów rozkładu
mu = np.linspace(0, 10, FEATURES).astype(np.float32)
sigma = np.linspace(0.5, 2.0, FEATURES).astype(np.float32)

# losowanie z rozkładu
X = np.random.normal(loc=mu, scale=sigma, size=(N, FEATURES)).astype(np.float32)

# dodanie szumu
noise = np.random.uniform(-NOISE, NOISE, size=(N,)).astype(np.float32)
y = X.mean(axis=1) + noise

print("X shape:", X.shape, "y shape:", y.shape)
print("Example y:", y[:5])
print(mu)
print

X shape: (1000000, 12) y shape: (1000000,)
Example y: [4.5140276 4.9083953 5.260459  5.5671186 5.0179157]
[ 0.          0.90909094  1.8181819   2.7272727   3.6363637   4.5454545
  5.4545455   6.3636365   7.2727275   8.181818    9.090909   10.        ]


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [39]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("R2:", r2)


MSE: 3.3320065995212644e-05
R2: 0.9997745752334595


In [40]:
bias = float(lr.intercept_)
weights = [float(w) for w in lr.coef_]

print("bias:", bias)
print("weights[:5]:", weights[:5], " ... total:", len(weights))

bias: -4.3392181396484375e-05
weights[:5]: [0.08334597945213318, 0.08332014083862305, 0.08332635462284088, 0.08332691341638565, 0.08333316445350647]  ... total: 12


In [41]:
model = {
    "bias": bias,
    "weights": weights,
    "features": FEATURES,
    "seed": SEED,
    "noise": NOISE,
    "note": "Trained on synthetic StreamGen-like data: label = mean(features) + uniform noise"
}

out_path = Path("linear_model.json")
out_path.write_text(json.dumps(model, indent=2))
print("Saved:", out_path.resolve())

Saved: /Users/konrad/Desktop/praca magisterska/spark-flink-kafka-benchmark-fixed/model/linear_model.json
